Criar uma tabela de Clientes para a equipe de marketing.
É importante somente pessoas que tenham e-mail e telefone cadastrados

In [0]:
bronze_path = '/Volumes/bikestore/bronze/bronze_volumes'
silver_path = '/Volumes/bikestore/silver/silver_volumes'

In [0]:
bronze_tables = {
    'tmp_bronze_brands' : f'{bronze_path}/brands/',
    'tmp_bronze_categories' : f'{bronze_path}/categories/',
    'tmp_bronze_customers' : f'{bronze_path}/customers/',
    'tmp_bronze_order_items' : f'{bronze_path}/order_items',
    'tmp_bronze_orders' : f'{bronze_path}/orders/',
    'tmp_bronze_products' : f'{bronze_path}/products/',
    'tmp_bronze_staffs' : f'{bronze_path}/staffs/',
    'tmp_bronze_stocks' : f'{bronze_path}/stocks/',
    'tmp_bronze_stores' : f'{bronze_path}/stores/'
}

for view_name, path in bronze_tables.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name))

In [0]:
%sql
SELECT 
    customers.customer_id,
    customers.first_name,
    customers.last_name,
    customers.phone,
    customers.email,
    customers.street,
    customers.city,
    customers.state,
    customers.zip_code
FROM tmp_bronze_customers customers
WHERE 1=1 -- Não muda nada, somente para organização 
AND customers.phone IS NOT NULL -- Vazio
AND customers.phone NOT IN ('NULL', 'NULL ', ' NULL') -- Texto NULL
AND customers.email IS NOT NULL -- Vazio
AND customers.email NOT IN ('NULL', 'NULL ', ' NULL') -- Texto NULL

In [0]:
df_customer_silver = spark.sql("""
SELECT 
    customers.customer_id,
    customers.first_name,
    customers.last_name,
    customers.phone,
    customers.email,
    customers.street,
    customers.city,
    customers.state,
    customers.zip_code
FROM tmp_bronze_customers customers
WHERE 1=1 -- Não muda nada, somente para organização 
AND customers.phone IS NOT NULL -- Vazio
AND customers.phone NOT IN ('NULL', 'NULL ', ' NULL') -- Texto NULL
AND customers.email IS NOT NULL -- Vazio
AND customers.email NOT IN ('NULL', 'NULL ', ' NULL') -- Texto NULL                 
                             """)

df_customer_silver.write.mode("overwrite").format("delta").option(
    "mergeSchema", "true"
).save(silver_path + "/customer")

In [0]:
%sql
-- Criando uma tabela física

CREATE TABLE bikestore.silver.silver_customer
USING DELTA
AS
SELECT *
FROM delta.`/Volumes/bikestore/silver/silver_volumes/customer`;